# PF4 Replication Analysis using FinnGen GWAS Summary Statistics

This notebook processes FinnGen GWAS summary statistics for heart failure, coronary artery disease, and myocardial infarction. FinnGen is used as an independent replication resource to check whether PF4-related variants also appear in comparable cardiovascular phenotypes.

In [1]:
from pathlib import Path
import json
import pandas as pd

pd.set_option("display.max_columns", None)

In [2]:
region_path = Path("../data/interim/ensembl/region.json")
variants_path = Path("../data/interim/ncbi/variants.csv")

finngen_hf_path = Path("../data/raw/finngen/finngen_heart_failure.tsv")
finngen_cad_path = Path("../data/raw/finngen/finngen_cad.tsv")
finngen_mi_path = Path("../data/raw/finngen/finngen_mi.tsv")

out_dir = Path("../data/interim/finngen")
out_dir.mkdir(parents=True, exist_ok=True)

out_hf_csv = out_dir / "heart_failure_associations.csv"
out_cad_csv = out_dir / "cad_associations.csv"
out_mi_csv = out_dir / "mi_associations.csv"
out_summary_json = out_dir / "summary.json"

In [3]:
with open(region_path, "r") as f:
    region = json.load(f)

variants_df = pd.read_csv(variants_path)

chromosome = str(region["chromosome"])
region_start = int(region["region_start"])
region_end = int(region["region_end"])

print("Gene:", region["gene_symbol"])
print("Assembly:", region["assembly_name"])
print("PF4 region:", f"chr{chromosome}:{region_start}-{region_end}")
print("Number of PF4 variants:", variants_df.shape[0])

variants_df.head()

Gene: PF4
Assembly: GRCh38
PF4 region: chr4:73930811-74032027
Number of PF4 variants: 2030


,rsID,Variant_type,Alleles_REF_ALT,CHRPOS_GRCh38,CHRPOS_GRCh37,Canonical_SPDI,Gene_name,Functional_consequence,Clinical_significance,Validation_flags,ALFA_freq,ALFA_AC,ALSPAC_freq,ALSPAC_AC,TOMMO_freq,TOMMO_AC,MAX_freq,Priority,Position_GRCh38,Location_relative_to_PF4
0,rs2233648,snv,C>G,4:73981419,4:74847136,NC_000004.12:73981418:C:G,PF4,"coding_sequence_variant,synonymous_variant",benign,"by-frequency,by-alfa,by-cluster",0.005184,244.0,0.0,0.0,0.000039,3.0,0.005184,Primary,73981419,inside_PF4
1,rs367951530,snv,G>A;G>T,4:73981468,4:74847185,"NC_000004.12:73981467:G:A,NC_000004.12:7398146...",PF4,"coding_sequence_variant,missense_variant",uncertain-significance,"by-frequency,by-alfa,by-cluster",0.000058,3.0,NaN,NaN,NaN,NaN,0.000058,Below threshold,73981468,inside_PF4
2,rs764474600,snv,C>G;C>T,4:73981460,4:74847177,"NC_000004.12:73981459:C:G,NC_000004.12:7398145...",PF4,"coding_sequence_variant,missense_variant",uncertain-significance,"by-frequency,by-alfa,by-cluster",0.000000,0.0,NaN,NaN,0.000013,1.0,0.000013,Below threshold,73981460,inside_PF4
3,rs765704070,snv,C>G;C>T,4:73981445,4:74847162,"NC_000004.12:73981444:C:G,NC_000004.12:7398144...",PF4,"coding_sequence_variant,missense_variant",uncertain-significance,"by-frequency,by-alfa,by-cluster",0.000000,0.0,NaN,NaN,NaN,NaN,0.000000,Below threshold,73981445,inside_PF4
4,rs982761496,snv,C>A,4:73981500,4:74847217,NC_000004.12:73981499:C:A,PF4,"coding_sequence_variant,missense_variant",uncertain-significance,"by-frequency,by-alfa",0.000000,0.0,NaN,NaN,NaN,NaN,0.000000,Below threshold,73981500,inside_PF4


In [4]:
for path in [finngen_hf_path, finngen_cad_path, finngen_mi_path]:
    if not path.exists():
        raise FileNotFoundError(f"FinnGen file not found: {path}")

In [5]:
pf4_variants_df = variants_df.copy()

pf4_variants_df["rsID"] = (
    pf4_variants_df["rsID"]
    .dropna()
    .astype(str)
)

pf4_rsids_set = set(
    pf4_variants_df["rsID"]
    .dropna()
    .astype(str)
    .unique()
)

priority_pf4_rsids_set = set(
    pf4_variants_df.loc[
        pf4_variants_df["Priority"].isin(["Primary", "Secondary"]),
        "rsID"
    ]
    .dropna()
    .astype(str)
    .unique()
)

print("Total PF4 rsIDs:", len(pf4_rsids_set))
print("Priority PF4 rsIDs:", len(priority_pf4_rsids_set))

Total PF4 rsIDs: 1976
Priority PF4 rsIDs: 176


In [6]:
finngen_preview = pd.read_csv(
    finngen_hf_path,
    sep="\t",
    nrows=5
)

finngen_preview

,#chrom,pos,ref,alt,rsids,nearest_genes,pval,mlogp,beta,sebeta,af_alt,af_alt_cases,af_alt_controls
0,1,13668,G,A,rs2691328,DDX11L1,0.529105,0.276458,0.061669,0.097985,0.005986,0.006138,0.005974
1,1,14506,G,A,rs1240557819,WASH7P,0.936567,0.028461,-0.008487,0.106644,0.004639,0.004614,0.004641
2,1,14521,C,T,rs1378626194,WASH7P,0.123522,0.908256,0.326778,0.212172,0.001083,0.001113,0.001081
3,1,14717,G,A,rs377122907,WASH7P,0.933506,0.029883,-0.036576,0.438374,0.000237,0.000244,0.000236
4,1,14773,C,T,rs878915777,WASH7P,0.788099,0.103419,0.015343,0.057084,0.014208,0.014380,0.014194


In [7]:
variant_annotation_cols = [
    "rsID",
    "MAX_freq",
    "Priority",
    "Variant_type",
    "Alleles_REF_ALT",
    "CHRPOS_GRCh38",
    "CHRPOS_GRCh37",
    "Functional_consequence",
    "Clinical_significance",
    "Location_relative_to_PF4"
]

variant_metadata = variants_df[variant_annotation_cols].drop_duplicates()

variant_metadata.head()

,rsID,MAX_freq,Priority,Variant_type,Alleles_REF_ALT,CHRPOS_GRCh38,CHRPOS_GRCh37,Functional_consequence,Clinical_significance,Location_relative_to_PF4
0,rs2233648,0.005184,Primary,snv,C>G,4:73981419,4:74847136,"coding_sequence_variant,synonymous_variant",benign,inside_PF4
1,rs367951530,0.000058,Below threshold,snv,G>A;G>T,4:73981468,4:74847185,"coding_sequence_variant,missense_variant",uncertain-significance,inside_PF4
2,rs764474600,0.000013,Below threshold,snv,C>G;C>T,4:73981460,4:74847177,"coding_sequence_variant,missense_variant",uncertain-significance,inside_PF4
3,rs765704070,0.000000,Below threshold,snv,C>G;C>T,4:73981445,4:74847162,"coding_sequence_variant,missense_variant",uncertain-significance,inside_PF4
4,rs982761496,0.000000,Below threshold,snv,C>A,4:73981500,4:74847217,"coding_sequence_variant,missense_variant",uncertain-significance,inside_PF4


In [8]:
def process_finngen_dataset(input_path, phenotype_label, endpoint_label):
    usecols = [
        "#chrom",
        "pos",
        "ref",
        "alt",
        "rsids",
        "nearest_genes",
        "pval",
        "mlogp",
        "beta",
        "sebeta",
        "af_alt",
        "af_alt_cases",
        "af_alt_controls"
    ]

    matched_chunks = []

    for chunk in pd.read_csv(
        input_path,
        sep="\t",
        usecols=usecols,
        chunksize=500_000,
        low_memory=False
    ):
        chunk = chunk.rename(columns={"#chrom": "chrom"})

        chunk["chrom"] = chunk["chrom"].astype(str)
        chunk["pos"] = pd.to_numeric(chunk["pos"], errors="coerce").astype("Int64")
        chunk["rsids"] = chunk["rsids"].fillna("").astype(str)

        region_chunk = chunk[
            (chunk["chrom"] == chromosome)
            & (chunk["pos"] >= region_start)
            & (chunk["pos"] <= region_end)
        ].copy()

        if region_chunk.empty:
            continue

        region_chunk["rsID"] = region_chunk["rsids"].str.split(",")

        region_chunk = region_chunk.explode("rsID")
        region_chunk["rsID"] = region_chunk["rsID"].astype(str).str.strip()

        matched = region_chunk[
            region_chunk["rsID"].isin(pf4_rsids_set)
        ].copy()

        if not matched.empty:
            matched_chunks.append(matched)

    if matched_chunks:
        finngen_matches_df = pd.concat(matched_chunks, ignore_index=True)
    else:
        finngen_matches_df = pd.DataFrame(columns=usecols + ["rsID"])

    finngen_matches_df = finngen_matches_df.drop_duplicates()

    finngen_matches_df = finngen_matches_df.rename(columns={
        "chrom": "FINNGEN_CHR",
        "pos": "FINNGEN_BP",
        "ref": "FINNGEN_ref",
        "alt": "FINNGEN_alt",
        "rsids": "FINNGEN_rsids",
        "nearest_genes": "FINNGEN_nearest_genes",
        "pval": "FINNGEN_p_value",
        "mlogp": "FINNGEN_mlogp",
        "beta": "FINNGEN_beta",
        "sebeta": "FINNGEN_std_error",
        "af_alt": "FINNGEN_alt_allele_freq",
        "af_alt_cases": "FINNGEN_alt_allele_freq_cases",
        "af_alt_controls": "FINNGEN_alt_allele_freq_controls"
    })

    finngen_matches_df["phenotype"] = phenotype_label
    finngen_matches_df["source_dataset"] = "FinnGen R12 summary statistics"
    finngen_matches_df["finngen_endpoint"] = endpoint_label

    numeric_cols = [
        "FINNGEN_BP",
        "FINNGEN_p_value",
        "FINNGEN_mlogp",
        "FINNGEN_beta",
        "FINNGEN_std_error",
        "FINNGEN_alt_allele_freq",
        "FINNGEN_alt_allele_freq_cases",
        "FINNGEN_alt_allele_freq_controls"
    ]

    for col in numeric_cols:
        finngen_matches_df[col] = pd.to_numeric(
            finngen_matches_df[col],
            errors="coerce"
        )

    finngen_matches_df = finngen_matches_df.sort_values(
        "FINNGEN_p_value",
        ascending=True,
        na_position="last"
    )

    finngen_annotated_df = finngen_matches_df.merge(
        variant_metadata,
        on="rsID",
        how="left"
    )

    return finngen_annotated_df

In [9]:
finngen_hf_df = process_finngen_dataset(
    input_path=finngen_hf_path,
    phenotype_label="Heart failure",
    endpoint_label="I9_HEARTFAIL"
)

finngen_hf_df.shape

(20, 26)

In [10]:
finngen_cad_df = process_finngen_dataset(
    input_path=finngen_cad_path,
    phenotype_label="Coronary artery disease",
    endpoint_label="I9_CHD"
)

finngen_cad_df.shape

(20, 26)

In [11]:
finngen_mi_df = process_finngen_dataset(
    input_path=finngen_mi_path,
    phenotype_label="Myocardial infarction",
    endpoint_label="I9_MI_STRICT"
)

finngen_mi_df.shape

(20, 26)

In [12]:
finngen_hf_df.to_csv(out_hf_csv, index=False)
finngen_cad_df.to_csv(out_cad_csv, index=False)
finngen_mi_df.to_csv(out_mi_csv, index=False)

print("Saved:", out_hf_csv)
print("Saved:", out_cad_csv)
print("Saved:", out_mi_csv)

Saved: ../data/interim/finngen/heart_failure_associations.csv
Saved: ../data/interim/finngen/cad_associations.csv
Saved: ../data/interim/finngen/mi_associations.csv


In [13]:
summary = {
    "source_dataset": "FinnGen R12 summary statistics",
    "genome_assembly": region["assembly_name"],
    "total_pf4_variants": int(len(pf4_rsids_set)),
    "priority_pf4_variants": int(len(priority_pf4_rsids_set)),
    "heart_failure": {
        "phenotype": "Heart failure",
        "finngen_endpoint": "I9_HEARTFAIL",
        "pf4_matches": int(len(finngen_hf_df)),
        "priority_matches": int(
            finngen_hf_df["Priority"].isin(["Primary", "Secondary"]).sum()
        )
    },
    "cad": {
        "phenotype": "Coronary artery disease",
        "finngen_endpoint": "I9_CHD",
        "pf4_matches": int(len(finngen_cad_df)),
        "priority_matches": int(
            finngen_cad_df["Priority"].isin(["Primary", "Secondary"]).sum()
        )
    },
    "mi": {
        "phenotype": "Myocardial infarction",
        "finngen_endpoint": "I9_MI_STRICT",
        "pf4_matches": int(len(finngen_mi_df)),
        "priority_matches": int(
            finngen_mi_df["Priority"].isin(["Primary", "Secondary"]).sum()
        )
    }
}

with open(out_summary_json, "w") as f:
    json.dump(summary, f, indent=4)

summary

{'source_dataset': 'FinnGen R12 summary statistics',
 'genome_assembly': 'GRCh38',
 'total_pf4_variants': 1976,
 'priority_pf4_variants': 176,
 'heart_failure': {'source_file': '../data/raw/finngen/finngen_heart_failure.tsv',
  'phenotype': 'Heart failure',
  'finngen_endpoint': 'I9_HEARTFAIL',
  'pf4_matches': 20,
  'priority_matches': 15},
 'cad': {'source_file': '../data/raw/finngen/finngen_cad.tsv',
  'phenotype': 'Coronary artery disease',
  'finngen_endpoint': 'I9_CHD',
  'pf4_matches': 20,
  'priority_matches': 15},
 'mi': {'source_file': '../data/raw/finngen/finngen_mi.tsv',
  'phenotype': 'Myocardial infarction',
  'finngen_endpoint': 'I9_MI_STRICT',
  'pf4_matches': 20,
  'priority_matches': 15}}